# Understanding Customer Sentiment on Twitter

**Author:** Alberto Diaz Durana
**Date:** March 2026
**Series:** AI Enhanced Productivity (W3 of 3)

## Objective

Explore the Customer Support on Twitter dataset using a multi-step AI workflow: Gemini API for automated exploration, AutoViz for visual analysis, and Hugging Face Transformers for sentiment classification. The goal is to evaluate where AI-assisted analysis accelerates understanding and where human judgment remains essential.

## Structure

1. Setup and Data Loading
2. Gemini Exploration (API-based dataset summary)
3. AutoViz Visualization
4. Sentiment Analysis (Hugging Face Transformers)
5. Synthesis and Reflections


In [1]:
# Cell 2 - Installs and Imports
import subprocess
import sys

packages = [
    'google-genai', 'autoviz', 'transformers', 'torch',
    'gdown', 'pandas', 'matplotlib', 'seaborn'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google import genai
from transformers import pipeline

print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__}")
print(f"Transformers: {__import__('transformers').__version__}")
print("All imports successful (AutoViz deferred to Phase 3)")


Python: 3.10.12 (main, Jan 26 2026, 14:55:28) [GCC 11.4.0]
Pandas: 2.1.4
Transformers: 5.3.0
All imports successful (AutoViz deferred to Phase 3)


In [9]:
# Cell 3 - Data Loading and Initial Inspection
import os
import gdown

# Colab-compatible path detection
try:
    import google.colab
    data_dir = '/content/data'
except ImportError:
    data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')

os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, 'twcs.csv')

if not os.path.exists(data_path):
    print("Downloading dataset from Google Drive...")
    gdown.download(id='1dvTN_tIwqd6eqncyWGzpTN788klfhKpR', output=data_path, quiet=False)
else:
    print(f"Dataset found locally: {data_path}")

df = pd.read_csv(data_path)

print(f"\nShape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumns and dtypes:")
print(df.dtypes)
print(f"\nSample rows:")
print(df.head(3).to_string())
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nInbound distribution:")
print(df['inbound'].value_counts())
print(f"\nInbound fraction: {df['inbound'].mean():.1%} inbound (customer) tweets")
print(f"\nTop 10 companies (by outbound tweet count):")
outbound = df[df['inbound'] == False]
print(outbound['author_id'].value_counts().head(10))
print(f"\nEmpty/missing text: {df['text'].isnull().sum() + (df['text'].str.strip() == '').sum()}")

Dataset found locally: /home/berto/_projects/AI-in-Data-Science/data/twcs.csv

Shape: 93 rows, 7 columns

Columns and dtypes:
tweet_id                     int64
author_id                   object
inbound                       bool
created_at                  object
text                        object
response_tweet_id           object
in_response_to_tweet_id    float64
dtype: object

Sample rows:
   tweet_id   author_id    inbound            created_at                                                                                text                                                                      response_tweet_id  in_response_to_tweet_id
0   119237         105834    True   Wed Oct 11 06:55:44 +0000 2017                                  @AppleSupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡       119236                   NaN        
1   119238   ChaseSupport   False   Wed Oct 11 13:25:49 +0000 2017  @105835 Your business means

### Phase 1 Observations: Dataset Structure

The dataset contains 93 tweets, a curated sample from the larger Customer Support on Twitter corpus. The split is nearly even: 49 inbound tweets (customer messages, 52.7%) and 44 outbound tweets (company responses, 47.3%). There is no missing text, which means every tweet carries usable content for analysis.

Company representation is uneven. AppleSupport accounts for 13 of the 44 outbound tweets (30%), followed by SpotifyCares and Tesco with 8 each. The remaining companies have between 1 and 4 responses. This concentration means sentiment patterns may reflect Apple-specific interactions more than a balanced industry view.

The response threading columns (`response_tweet_id` and `in_response_to_tweet_id`) have missing values (28 and 25 respectively), which is expected: not every tweet is part of a reply chain. These columns are useful for reconstructing conversations but are not needed for sentiment classification.

With only 49 inbound tweets, there is no need for sampling. We can run sentiment analysis on the full set of customer messages and still inspect results individually.


In [4]:
# Cell 4 - Phase 2: Gemini Exploration
# Load API key (Colab userdata first, local .env fallback)
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
except (ImportError, Exception):
    env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.strip().startswith('GOOGLE_API_KEY='):
                    api_key = line.strip().split('=', 1)[1]

if not api_key:
    api_key = os.environ.get('GOOGLE_API_KEY')

assert api_key, "GOOGLE_API_KEY not found. Set it in .env, Colab Secrets, or as an environment variable."

client = genai.Client(api_key=api_key)

# Build a concise dataset summary for the prompt
dataset_summary = f"""Dataset: Customer Support on Twitter
Shape: {df.shape[0]} rows, {df.shape[1]} columns
Columns and types:
{df.dtypes.to_string()}

Missing values:
{df.isnull().sum().to_string()}

Inbound (customer) tweets: {df['inbound'].sum()} ({df['inbound'].mean():.1%})
Outbound (company) tweets: {(~df['inbound']).sum()} ({(~df['inbound']).mean():.1%})

Sample inbound tweets:
{df[df['inbound'] == True]['text'].head(5).to_string()}

Sample outbound tweets:
{df[df['inbound'] == False]['text'].head(5).to_string()}
"""

prompt = f"""You are a data analyst. Given the following dataset summary, please:
1. Describe the dataset structure and key characteristics
2. Identify notable patterns or observations
3. Suggest analytical directions, particularly for sentiment analysis of customer tweets

{dataset_summary}"""

print("Sending dataset summary to Gemini...\n")
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt
)

print("=== Gemini's Dataset Analysis ===\n")
print(response.text)


Sending dataset summary to Gemini...

=== Gemini's Dataset Analysis ===

As a data analyst, here's an analysis of the provided dataset summary:

---

## Data Analysis Report: Customer Support on Twitter

### 1. Dataset Structure and Key Characteristics

*   **Dataset Size:** The dataset is relatively small, consisting of 93 rows and 7 columns. This size is suitable for initial exploratory analysis but might be limiting for complex machine learning models without additional data.
*   **Columns and Types:**
    *   `tweet_id` (int64): Unique identifier for each tweet. No missing values.
    *   `author_id` (object): Identifier for the tweet's author. No missing values. The sample shows anonymized IDs (e.g., `@105835`), indicating privacy measures.
    *   `inbound` (bool): A crucial indicator, `True` for customer tweets and `False` for company (support) tweets. No missing values.
    *   `created_at` (object): Timestamp of tweet creation. No missing values. This will need to be converted

### Phase 2 Evaluation: Gemini as an Exploration Tool

Gemini provided a structured, thorough overview of the dataset without writing any code. In a single prompt, it identified the key structural elements (conversation threading via `response_tweet_id` and `in_response_to_tweet_id`), recognized the inbound/outbound split, and flagged practical details like `created_at` needing datetime conversion and `in_response_to_tweet_id` being float64 due to NaN values.

The analytical suggestions are relevant but broad. Gemini recommended VADER and TextBlob for sentiment scoring; our plan uses Hugging Face Transformers instead, which provides pre-trained models better suited for nuanced social media text. The suggestion to reconstruct conversation threads and track sentiment evolution is valuable, though with only 93 rows the analytical depth is limited.

Where Gemini adds the most value is in the initial orientation phase: understanding what the data contains, what questions are worth asking, and what preprocessing is needed, all before writing a single line of analysis code. For a larger or unfamiliar dataset, this kind of rapid overview could save significant exploration time.

**Limitations observed:** The suggestions are generic enough to apply to most customer support datasets. Gemini did not flag anything surprising or counterintuitive, which is expected given the small sample and straightforward structure. The tool works best as a starting accelerator, not as a substitute for domain-specific analysis.


In [5]:
# Cell 5 - Phase 3: AutoViz Visualization
import warnings

# Suppress all warnings for this cell (AutoViz + pandas_dq compatibility issues)
_original_warn = warnings.warn
warnings.warn = lambda *a, **kw: None

# Fix IPython/matplotlib backend conflict
import IPython.core.pylabtools as _pylabtools
_pylabtools.backend2gui = {}

# Add derived numeric features for visualization
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

# Prepare a subset with visualizable columns
viz_df = df[['author_id', 'inbound', 'text_length', 'word_count']].copy()
viz_df['inbound'] = viz_df['inbound'].astype(str)

from autoviz import AutoViz_Class

AV = AutoViz_Class()
report = AV.AutoViz(
    filename='',
    dfte=viz_df,
    depVar='',
    verbose=1,
    lowess=False,
    chart_format='svg',
    max_rows_analyzed=100,
    max_cols_analyzed=10
)

# Restore warnings
warnings.warn = _original_warn

print(f"\nAutoViz completed. Columns analyzed: {list(viz_df.columns)}")


Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)
Shape of your Data Set loaded: (93, 4)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  0
    Number of Integer-Categorical Columns =  2
    Number of String-Categorical Columns =  1
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  1
    Number of Numeric-Boolean Columns =  0
    Number of Discrete String Columns =  0
    Number of NLP String Columns =  0
    Number of Da

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
author_id,object,0.000000,46,,,No issue
inbound,object,0.000000,2,,,No issue
text_length,int64,0.000000,68,32.000000,170.000000,No issue
word_count,int64,0.000000,26,4.000000,31.000000,"Column has 1 outliers greater than upper bound (37.00) or lower than lower bound(5.00). Cap them or remove them., Column has a high correlation with ['text_length']. Consider dropping one of them."


Number of All Scatter Plots = 3
All Plots done
Time to run AutoViz = 1 seconds 

 ###################### AUTO VISUALIZATION Completed ########################

AutoViz completed. Columns analyzed: ['author_id', 'inbound', 'text_length', 'word_count']


### Phase 3 Evaluation: AutoViz for Quick Visualization

AutoViz classified the four columns automatically and produced scatter plots in one second. The data quality report flagged three practical findings: 3 duplicate rows, 1 outlier in `word_count` (a tweet with only 4 words, below the lower bound of 5), and high correlation between `text_length` and `word_count`, which is expected since longer tweets naturally contain more words.

For a text-heavy dataset like this, AutoViz's value is limited. Most columns are categorical or text-based, leaving only the two derived numeric features (`text_length`, `word_count`) for visualization. The tool is better suited to tabular datasets with many numeric features, as in the W2 housing project. Here it confirms what we already know: tweet lengths cluster in the 80-170 character range with moderate word counts, and the inbound/outbound split is roughly even.

The main takeaway is the duplicate detection, which we would otherwise need to check manually. For the sentiment analysis phase, these duplicates should be noted but are unlikely to affect results given the small sample size.


In [6]:
# Cell 6 - Phase 4: Sentiment Analysis with Hugging Face
import re
from transformers import pipeline
import torch

# Load a sentiment model trained on tweets (3 classes: negative, neutral, positive)
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    device = 0 if torch.cuda.is_available() else -1
)

# Filter to inbound (customer) tweets
inbound_df = df[df['inbound'] == True].copy()
print(f"Analyzing sentiment for {len(inbound_df)} inbound (customer) tweets\n")

# Clean tweets: remove @mentions for better classification
def clean_tweet(text):
    text = re.sub(r'@\S+', '', text).strip()
    return text

inbound_df['clean_text'] = inbound_df['text'].apply(clean_tweet)

# Run sentiment analysis
results = sentiment_pipe(inbound_df['clean_text'].tolist(), truncation=True)

inbound_df['sentiment'] = [r['label'] for r in results]
inbound_df['confidence'] = [round(r['score'], 3) for r in results]

# Distribution
print("=== Sentiment Distribution ===")
print(inbound_df['sentiment'].value_counts())
print(f"\nMean confidence: {inbound_df['confidence'].mean():.3f}")

# Show sample results
print("\n=== Sample Results ===")
for _, row in inbound_df.head(10).iterrows():
    label = row['sentiment']
    conf = row['confidence']
    text = row['text'][:80]
    print(f"  [{label:8s} {conf:.2f}] {text}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Analyzing sentiment for 49 inbound (customer) tweets

=== Sentiment Distribution ===
sentiment
negative    27
neutral     14
positive     8
Name: count, dtype: int64

Mean confidence: 0.782

=== Sample Results ===
  [negative 0.85] @AppleSupport causing the reply to be disregarded and the tapped notification un
  [negative 0.60] @76328 I really hope you all change but I'm sure you won't! Because you don't ha
  [negative 0.67] @VirginTrains see attached error message. I've tried leaving a voicemail several
  [neutral  0.73] @VirginTrains yep, I've tried laptop too several times over the past week and ag
  [negative 0.91] @VirginTrains I still haven't heard &amp; the number I'm directed to by phone is
  [negative 0.77] @105838 @AppleSupport Me too am suffering , hope the can find a solution
  [negative 0.92] @AppleSupport hi #apple, I’ve a concern about the latest ios is too slow on #iph
  [negative 0.96] I just updated my phone and suddenly everything takes ages to load wtf @76099 th
  

### Phase 4 Results: Sentiment Analysis

The Hugging Face model (`cardiffnlp/twitter-roberta-base-sentiment-latest`), a RoBERTa model fine-tuned specifically on tweets, classified all 49 inbound customer tweets in seconds. The distribution skews negative, which is expected: people contact support when something is wrong.

| Sentiment | Count | Share |
|-----------|-------|-------|
| Negative  | 27    | 55%   |
| Neutral   | 14    | 29%   |
| Positive  | 8     | 16%   |

Mean confidence is 0.78, indicating the model is reasonably sure of its classifications. Spot-checking confirms the labels are sensible: complaints about iOS slowness and unresponsive support lines are flagged as negative (0.91-0.96 confidence), while a "Thanks!" reply to Spotify is positive. The lower-confidence predictions (0.56-0.61) tend to appear on ambiguous tweets where the tone is informational rather than clearly emotional.

Preprocessing was minimal: only @mention removal, since the model was trained on tweet-style text and handles abbreviations, hashtags, and informal language natively. No tokenization, stemming, or stop-word removal was needed, which is a significant productivity advantage over classical NLP pipelines.


In [7]:
# Cell 7 - Sentiment Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sentiment distribution bar chart
colors = {'negative': '#e74c3c', 'neutral': '#95a5a6', 'positive': '#2ecc71'}
counts = inbound_df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
axes[0].set_title('Sentiment Distribution (Inbound Tweets)')
axes[0].set_ylabel('Count')
for i, (label, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + 0.3, str(val), ha='center', fontweight='bold')

# Confidence distribution by sentiment
for sentiment in ['negative', 'neutral', 'positive']:
    subset = inbound_df[inbound_df['sentiment'] == sentiment]['confidence']
    if len(subset) > 0:
        axes[1].hist(subset, bins=10, alpha=0.6, label=sentiment, color=colors[sentiment])
axes[1].set_title('Model Confidence by Sentiment')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nConfidence summary by sentiment:")
print(inbound_df.groupby('sentiment')['confidence'].describe()[['mean', 'min', 'max']].round(3).to_string())



Confidence summary by sentiment:
           mean    min    max 
sentiment                     
negative   0.827  0.528  0.957
neutral    0.716  0.469  0.856
positive   0.746  0.504  0.990


In [8]:
# Cell 8 - Sentiment by Company
# Match inbound tweets to the company they're addressing (outbound author)
# Use in_response_to_tweet_id to find which company thread the customer is in
company_map = df[df['inbound'] == False].set_index('tweet_id')['author_id']
inbound_df['company'] = inbound_df['in_response_to_tweet_id'].map(company_map)

# Some inbound tweets initiate threads (no in_response_to), try response_tweet_id
mask = inbound_df['company'].isna()
response_map = df[df['inbound'] == False].set_index('tweet_id')['author_id']
inbound_df.loc[mask, 'company'] = inbound_df.loc[mask, 'response_tweet_id'].map(response_map)

# Fill remaining with the @mention in the tweet text
import re
def extract_mention(text):
    match = re.search(r'@(\w+)', text)
    return match.group(1) if match else 'Unknown'

mask = inbound_df['company'].isna()
inbound_df.loc[mask, 'company'] = inbound_df.loc[mask, 'text'].apply(extract_mention)

# Crosstab: company vs sentiment
cross = pd.crosstab(inbound_df['company'], inbound_df['sentiment'])
cross = cross.reindex(columns=['negative', 'neutral', 'positive'], fill_value=0)
cross['total'] = cross.sum(axis=1)
cross = cross.sort_values('total', ascending=False)

print("=== Sentiment by Company ===")
print(cross.to_string())

# Plot for top companies (at least 3 tweets)
top = cross[cross['total'] >= 3].drop(columns='total')
if len(top) > 0:
    colors_list = ['#e74c3c', '#95a5a6', '#2ecc71']
    top.plot(kind='barh', stacked=True, color=colors_list, figsize=(10, 4))
    plt.title('Sentiment by Company (3+ customer tweets)')
    plt.xlabel('Number of Tweets')
    plt.tight_layout()
    plt.show()


=== Sentiment by Company ===
sentiment        negative  neutral  positive  total
company                                            
AppleSupport         7        1         1       9  
Tesco                3        5         0       8  
76099                6        0         1       7  
SpotifyCares         2        1         4       7  
VirginTrains         2        1         0       3  
British_Airways      1        1         0       2  
105838               1        0         0       1  
HPSupport            1        0         0       1  
Unknown              1        0         0       1  
SouthwestAir         0        0         1       1  
O2                   0        1         0       1  
Ask_Spectrum         0        1         0       1  
105860               0        1         0       1  
77245                1        0         0       1  
76803                0        1         0       1  
76501                0        0         1       1  
76495                1        0    

### Phase 5: Synthesis and Reflections

**What the tools revealed**

The three AI-assisted tools each contributed a different layer of understanding. Gemini provided a rapid, zero-code orientation to the dataset: structure, missing values, and analytical directions, all in a single prompt. AutoViz confirmed the dataset's limited numeric profile and flagged duplicates, but added little beyond what manual inspection would find on a 93-row, text-heavy dataset. Hugging Face Transformers delivered the core analytical value: a pre-trained, tweet-specific sentiment model that classified all 49 customer messages with no labeled training data and no custom preprocessing.

**Key findings**

Customer support tweets skew negative (55%), which aligns with the nature of support interactions. The model's confidence is highest for negative tweets (mean 0.83), suggesting complaints are linguistically more distinct than neutral or positive messages. At the company level, AppleSupport receives the most negative sentiment (7 of 9 tweets), while SpotifyCares stands out as the only company with a positive lean (4 of 7 tweets). Tesco's interactions are predominantly neutral (5 of 8), possibly reflecting factual exchanges rather than emotional complaints.

**Productivity impact**

The entire analysis, from data loading to company-level sentiment breakdown, required minimal custom code. The Hugging Face pipeline abstracted away tokenization, model loading, and inference into a single function call. Gemini replaced the initial manual inspection phase. The main time investment was not in analysis but in resolving tool compatibility issues: API quota limits, deprecated model names, and library version conflicts (AutoViz/IPython/matplotlib). This is a realistic reflection of AI-assisted workflows: the tools accelerate the analytical work, but environment setup and dependency management remain manual bottlenecks.

**Limitations**

The 93-row sample limits generalizability. Company-level patterns are based on as few as 3 tweets, which is insufficient for reliable conclusions. The sentiment model was not validated against human labels for this specific dataset, so misclassifications are possible, particularly for the lower-confidence predictions (neutral tweets at 0.72 mean confidence). A larger dataset would allow for statistical significance testing and more robust company comparisons, which is the motivation for the planned challenge notebook using the full 2.8M-tweet corpus.
